# QInstruments BioShake

The BioShake is a family of heater-cooler-shaker devices from QInstruments. Depending on the model, it supports:

- [Shaking](../../capabilities/shaking) (orbital, 200--5000 rpm)
- [Temperature control](../../capabilities/temperature-control) (heating, and active cooling on Q1/Q2/ColdPlate)
- Plate locking (ELM models)

All models share the same serial firmware, so a single driver covers the entire family. Use a **model-specific factory function** to create instances -- it pre-fills dimensions and enables only the capabilities your hardware supports.

| Model | PLR Name | Shaking (rpm) | Plate Lock | Heating | Active Cooling |
|---|---|---|---|---|---|
| BioShake Q1 | `BioShakeQ1` | 200--3000 | yes | yes | yes |
| BioShake Q2 | `BioShakeQ2` | 200--3000 | yes | yes | yes |
| BioShake 3000 | `BioShake3000` | 200--3000 | no | no | no |
| BioShake 3000 elm | `BioShake3000Elm` | 200--3000 | yes | no | no |
| BioShake 3000 elm DWP | `BioShake3000ElmDWP` | 200--3000 | yes | no | no |
| BioShake D30 elm | `BioShakeD30Elm` | 200--2000 | yes | no | no |
| BioShake 5000 elm | `BioShake5000Elm` | 200--5000 | yes | no | no |
| BioShake 3000-T | `BioShake3000T` | 200--3000 | no | yes | no |
| BioShake 3000-T elm | `BioShake3000TElm` | 200--3000 | yes | yes | no |
| BioShake D30-T elm | `BioShakeD30TElm` | 200--2000 | yes | yes | no |
| Heatplate | `Heatplate` | -- | no | yes | no |
| ColdPlate | `ColdPlate` | -- | no | yes | yes |

See the [BioShake integration manual](https://www.qinstruments.com/fileadmin/Article/All/integration-manual-en-1-8-0.pdf) for hardware setup. Connect via RS232 or USB-A with a 24 VDC power supply.

## Setup

In [ ]:
from pylabrobot.qinstruments import BioShakeQ1

bs = BioShakeQ1(name="bioshake", port="COM1")  # replace with your port
await bs.setup()

By default, `setup()` resets the device and homes the shaker. Pass `skip_home=True` to skip this.

## Shaking

The BioShake exposes a {class}`~pylabrobot.capabilities.shaking.shaking.Shaker` on `bs.shaker`. For the full API, see [Shaking](../../capabilities/shaking).

In [ ]:
# Shake for 60 seconds at 500 rpm (blocks until done)
await bs.shaker.shake(speed=500, duration=60)

In [ ]:
# Or start/stop manually
await bs.shaker.shake(speed=300)
# ... do other things ...
await bs.shaker.stop_shaking()

BioShake supports acceleration and deceleration ramps (seconds to reach full speed). Pass these as backend params:

In [ ]:
await bs.shaker.shake(speed=500, acceleration=3)
await bs.shaker.stop_shaking(deceleration=5)

Lock and unlock the plate (ELM models):

In [ ]:
await bs.shaker.lock_plate()
await bs.shaker.shake(speed=1000, duration=10)
await bs.shaker.unlock_plate()

## Temperature control

Models with heating/cooling expose a {class}`~pylabrobot.capabilities.temperature_controlling.temperature_controller.TemperatureController` on `bs.tc`. For the full API, see [Temperature Control](../../capabilities/temperature-control).

In [ ]:
await bs.tc.set_temperature(37.0)
await bs.tc.wait_for_temperature(tolerance=0.5)

In [ ]:
current = await bs.tc.request_temperature()
print(f"{current:.1f} \u00b0C")

In [ ]:
await bs.tc.deactivate()

## Multiple devices

When using multiple BioShake devices, run them concurrently with `asyncio.gather`:

In [ ]:
import asyncio
from pylabrobot.qinstruments import BioShake3000Elm

bs1 = BioShake3000Elm(name="bs1", port="COM1")
bs2 = BioShake3000Elm(name="bs2", port="COM2")

await asyncio.gather(bs1.setup(), bs2.setup())
await asyncio.gather(
    bs1.shaker.shake(speed=500, duration=30),
    bs2.shaker.shake(speed=500, duration=30),
)

## Teardown

In [ ]:
await bs.stop()